## Importing required libraries


In [ ]:
import pandas as pd
from astropy.time import Time
import matplotlib.pyplot as plt
import os

- Uploading the Excel file of the tiles, downloaded from the CRTS website.
- Converting from MJD to datetime.

In [2]:
df = pd.read_csv("km3netevent_lightcurve_tiles.csv")

t = Time(df['MJD'], format='mjd')
df['date'] = t.to_datetime()
df.head()

,MasterID,Mag,Magerr,RA,Dec,MJD,Blend,date
0,1.007030e+12,20.172,0.389,94.29311,-7.84727,54830.27443,0,2008-12-30 06:35:10.752
1,1.007030e+12,19.705,0.300,94.29386,-7.84747,54830.27983,0,2008-12-30 06:42:57.312
2,1.007030e+12,19.147,0.225,94.29360,-7.84727,54830.29053,0,2008-12-30 06:58:21.792
3,1.007030e+12,20.453,0.457,94.29320,-7.84716,54830.29593,0,2008-12-30 07:06:08.352
4,1.007030e+12,19.896,0.301,94.29340,-7.84745,55157.44256,0,2009-11-22 10:37:17.184


## Light Curve Generation

This section reads the processed light-curve data, converts observation times from MJD to datetime format, and generates individual light-curve plots for each source (`MasterID`). The resulting figures are automatically saved for later analysis.

In [3]:
df = pd.read_csv("km3netevent_lightcurve_tiles.csv")
plt.style.use('default')
# clean column names
df.columns = df.columns.str.strip()


t = Time(df['MJD'], format='mjd')
df['date'] = t.to_datetime()

# Make folder for plots
os.makedirs("lightcurve_plots", exist_ok=True)

for master_id, group in df.groupby("MasterID"):

    date = group["date"]
    mag = group["Mag"]
    err = group["Magerr"]

    plt.figure(figsize=(10,6), facecolor='white')

    plt.errorbar(date, mag, yerr=err,
                 fmt='o', markersize=4, capsize=2)

    plt.xlabel("Date (year)", color='black')
    plt.ylabel("V magnitude", color='black')
    plt.title(f"Light Curve - MasterID {master_id}", color='black')

    ax = plt.gca()
    ax.tick_params(axis='both', colors='black')

    for spine in ax.spines.values():
        spine.set_color('black')

    plt.gca().invert_yaxis()
    plt.grid(True, color='white', alpha=0.1)

    plt.savefig(f"lightcurve_plots/lightcurve_{master_id}.png", dpi=300)
    plt.close()

**This following section processes each light curve individually and applies a 3-sigma filtering procedure to identify and remove potential outliers.**

The workflow is:

1. Load and clean the light-curve dataset.
2. Group observations by `MasterID`.
3. Exclude sources with fewer than 10 observations.
4. Compute the mean and standard deviation of the magnitude values.
5. Apply a 3-sigma filter to remove measurements outside the range:

   mean ± 3σ

6. Compare the original and sigma-corrected light curves.
7. Save comparison plots for sources where the filtering modifies the data.
8. Save the original light curve when no points are removed.

This step removes measurements that fall outside the 3-sigma range and allows the resulting light curves to be compared with the original data.

## Data Preparation and Three-Sigma Filtering


In [ ]:
# READ CSV FILE
file_path = "km3netevent_lightcurve_tiles.csv"

df = pd.read_csv(file_path, header=None, dtype=str)

# ADD COLUMN NAMES
df.columns = ["MasterID", "Mag", "Magerr", "RA", "Dec", "MJD", "Blend"]

# CLEAN DATA
df["MasterID"] = df["MasterID"].str.strip()

df["Mag"] = pd.to_numeric(df["Mag"], errors="coerce")
df["Magerr"] = pd.to_numeric(df["Magerr"], errors="coerce")
df["MJD"] = pd.to_numeric(df["MJD"], errors="coerce")

# Remove bad rows
df = df.dropna(subset=["MasterID", "Mag", "Magerr", "MJD"])

# CONVERT MJD TO DATE
t = Time(df["MJD"], format="mjd")
df["date"] = t.to_datetime()

# DEBUG INFO
counts = df["MasterID"].value_counts()

print("Total MasterIDs:", df["MasterID"].nunique())
print("MasterIDs with >=10 rows:", (counts >= 10).sum())

# CREATE OUTPUT FOLDER
output_folder = "lightcurve_plots_sigma"

os.makedirs(output_folder, exist_ok=True)

# LOOP THROUGH MASTERIDS
for master_id, group in df.groupby("MasterID"):

    # Skip MasterIDs with fewer than 10 rows
    if len(group) < 10:
        continue

    # Sort by date
    group = group.sort_values("date")

    # THREE SIGMA CORRECTION
    avg = group["Mag"].mean()
    std = group["Mag"].std()

    lower = avg - 3 * std
    upper = avg + 3 * std

    group_clean = group[(group["Mag"] >= lower) & (group["Mag"] <= upper)]

    # Check if sigma correction changed anything
    changed = len(group_clean) < len(group)

    # Safe filename
    filename_id = str(master_id).replace("/", "_")


## Generate Original and Sigma-Corrected Light Curve Plots

In [4]:
    # IF DATA CHANGED -> SUBPLOTS
    if changed:

        fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

        # ORIGINAL PLOT
        axes[0].errorbar(group["date"], group["Mag"], yerr=group["Magerr"], fmt='o', markersize=4, capsize=2)

        axes[0].set_title(f"Original Light Curve - MasterID {master_id}")

        axes[0].set_ylabel("Magnitude")

        axes[0].invert_yaxis()

        axes[0].grid(True, alpha=0.3)

        # SIGMA CORRECTED PLOT
        axes[1].errorbar(group_clean["date"], group_clean["Mag"], yerr=group_clean["Magerr"], fmt='o', markersize=4, capsize=2)

        axes[1].set_title(f"3 Sigma Corrected - MasterID {master_id}")

        axes[1].set_xlabel("Date")
        axes[1].set_ylabel("Magnitude")

        axes[1].invert_yaxis()

        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()

        plt.savefig(f"{output_folder}/lightcurve_{filename_id}_comparison.png", dpi=300)

        plt.close()

    # IF NO CHANGE -> ORIGINAL ONLY
    else:

        plt.figure(figsize=(10, 6))

        plt.errorbar(group["date"], group["Mag"], yerr=group["Magerr"], fmt='o', markersize=4, capsize=2)

        plt.title(f"Light Curve - MasterID {master_id}")

        plt.xlabel("Date")
        plt.ylabel("Magnitude")

        plt.gca().invert_yaxis()

        plt.grid(True, alpha=0.3)

        plt.tight_layout()

        plt.savefig(f"{output_folder}/lightcurve_{filename_id}.png", dpi=300)

        plt.close()

# DONE
print("Done.")
print("Plots saved in:", output_folder)

Total MasterIDs: 98
MasterIDs with >=10 rows: 91
Done.
Plots saved in: lightcurve_plots_sigma
